In [1]:
import pandas as pd
import re, base64

SAFE_QP = re.compile(r'=([2-7][0-9A-F])')  

def decode_email_text(text):
    text = str(text)

    stripped = text.strip()
    if len(stripped) > 60 and ' ' not in stripped[:60] and re.fullmatch(r'[A-Za-z0-9+/=\s]+', stripped):
        try:
            raw = re.sub(r'\s', '', stripped)
            raw += '=' * (-len(raw) % 4)
            decoded = base64.b64decode(raw).decode('utf-8', errors='ignore')
            if sum(c.isalpha() or c.isspace() for c in decoded) / max(len(decoded), 1) > 0.8:
                text = decoded
        except Exception:
            pass

    if len(SAFE_QP.findall(text)) >= 3:
        text = re.sub(r'=\r?\n', '', text)
        text = SAFE_QP.sub(lambda m: chr(int(m.group(1), 16)), text)

    return text

df = pd.read_csv('../data/processed/master_dataset.csv')
df['message_decoded'] = df['message'].apply(decode_email_text)

pat = r'=[0-9A-F]{2}'
print("Artifacts before:", df['message'].str.contains(pat, regex=True).sum())
print("Artifacts after: ", df['message_decoded'].str.contains(pat, regex=True).sum())

changed = df[df['message'] != df['message_decoded']]
print("Rows changed:", len(changed))
for _, r in changed.sample(3, random_state=2).iterrows():
    print('BEFORE:', str(r['message'])[:150].replace('\n', ' '))
    print('AFTER: ', str(r['message_decoded'])[:150].replace('\n', ' '))
    print('---')

Artifacts before: 2274
Artifacts after:  1426
Rows changed: 1333
BEFORE: from Mrs=2ELydia JohnsonLiberte Sicap 1=2CDakar Senegal=2EGoodday Dear=2CI know this mail will come to you as a surprise=2Cnevertheless I am Mrs=2E Ly
AFTER:  from Mrs.Lydia JohnsonLiberte Sicap 1,Dakar Senegal.Goodday Dear,I know this mail will come to you as a surprise,nevertheless I am Mrs. Lydia Johnson,
---
BEFORE: Assalam aalaikoom=2CI am Abdul Ramatd=2EI am married to Late Kazeem Abdulof blessed memory who is an oil explorer in Singaporeand Kuwait for twelve ye
AFTER:  Assalam aalaikoom,I am Abdul Ramatd.I am married to Late Kazeem Abdulof blessed memory who is an oil explorer in Singaporeand Kuwait for twelve years 
---
BEFORE: Investment proposal from=3A Dr=2E Tunde Cole=2E Block 4 Federal Secretariat Complex=2C #16 Norman Williams Way=2E Ikoyi-Lagos ATTENTION=3A REQUEST FOR
AFTER:  Investment proposal from: Dr. Tunde Cole. Block 4 Federal Secretariat Complex, #16 Norman Williams Way. Ikoyi-Lagos ATTENTIO

In [2]:
from collections import Counter

WS_QP = re.compile(r'=(0[9AD])')   # tab, line feed, carriage return

def decode_email_text_v2(text):
    return WS_QP.sub(' ', decode_email_text(text))

df['message_decoded'] = df['message'].apply(decode_email_text_v2)

pat = r'=[0-9A-F]{2}'
print("Rows still matching:", df['message_decoded'].str.contains(pat, regex=True).sum())

left = Counter(re.findall(pat, ' '.join(df['message_decoded'])))
print(left.most_common(15))

Rows still matching: 1378
[('=E9', 2040), ('=92', 499), ('=E8', 352), ('=E0', 343), ('=A0', 251), ('=02', 221), ('=A3', 176), ('=20', 151), ('=3D', 135), ('=F4', 121), ('=10', 105), ('=C2', 95), ('=A1', 79), ('=00', 78), ('=EA', 60)]


In [3]:
import unicodedata

ALL_QP = re.compile(r'=([0-9A-F]{2})')

def decode_email_text_v3(text):
    text = decode_email_text_v2(text)
    if len(ALL_QP.findall(text)) >= 3:
        text = ALL_QP.sub(lambda m: chr(int(m.group(1), 16)), text)
    text = unicodedata.normalize('NFKD', text)
    return ''.join(c for c in text if not unicodedata.combining(c))

df['message_decoded'] = df['message'].apply(decode_email_text_v3)

RULES = [  # (category, min distinct hits needed, patterns) - checked in this order
    ('digital_arrest', 1, [r'digital arrest', r'enforcement directorate', r'\bcbi\b',
        r'narcotics (bureau|control)', r'arrest warrant', r'money laundering case',
        r'cyber ?crime (cell|branch|police)']),
    ('upi_payment', 1, [r'\bupi\b', r'\bgpay\b', r'google ?pay', r'phone ?pe\b',
        r'\bpaytm\b', r'\bbhim\b', r'collect request']),
    ('banking_kyc', 1, [r'\bkyc\b', r'pan card', r'\baadhaar\b|\baadhar\b', r'net ?banking',
        r'account (has been|will be|is) (blocked|suspended|frozen|deactivated)',
        r'(debit|credit) card (has been|will be|is) (blocked|expired|suspended)',
        r'verify your (account|identity|details)']),
    ('investment', 2, [r'guaranteed (returns|profit)', r'double your (money|investment)',
        r'crypto(currency)?', r'bitcoin', r'forex', r'trading (platform|group|bot|signals)',
        r'investment (plan|scheme|opportunity)', r'passive income', r'monthly (returns|profit)']),
    ('lottery', 2, [r'lottery', r'\b(you|u) (have )?(just )?won\b', r'\bwinner\b', r'prize',
        r'jackpot', r'claim (your|ur)\b', r'claim code', r'awarded', r'congratulations']),
    ('advance_fee', 3, [r'next of kin', r'late (husband|father|mother|client|president)',
        r'deceased', r'plane crash|car accident|air crash',
        r'business (proposal|transaction|partnership|relationship)',
        r'transfer (of|the|this) (the )?(sum|fund|money)', r'\bmillion\b',
        r'foreign (account|partner)', r'beneficiary', r'dear (friend|sir|beloved)',
        r'confidential|secrecy', r'central bank of nigeria', r'barrister|attorney',
        r'\d+\s?% (of|for) (the )?(total|fund|sum)', r'abidjan|lagos|abuja|dakar|accra']),
]

def assign_type(row):
    if row['label'] != 'scam':
        return 'none'
    if row['source'] == 'phishing_email':
        return 'phishing'
    if row['source'] == 'job_posting':
        return 'job_scam'
    text = row['message_decoded'].lower()
    for cat, min_hits, pats in RULES:
        if sum(bool(re.search(p, text)) for p in pats) >= min_hits:
            return cat
    return 'unknown'

df['scam_type_new'] = df.apply(assign_type, axis=1)

print(pd.crosstab(df['source'], df['scam_type_new']))
print()
print(pd.crosstab(df['scam_type'], df['scam_type_new']))   # old labels (rows) vs new (columns)

for cat in ['advance_fee', 'lottery', 'investment', 'banking_kyc']:
    sub = df[df['scam_type_new'] == cat]
    print(f"\n===== {cat} ({len(sub)} rows) =====")
    for m in sub.sample(min(5, len(sub)), random_state=1)['message_decoded']:
        print('-', str(m)[:150].replace('\n', ' '))

scam_type_new   advance_fee  banking_kyc  job_scam  lottery   none  phishing  \
source                                                                         
fraud_email            2577            4         0       16   5913         0   
job_posting               0            0       718        0  15348         0   
phishing_email            0            0         0        0  10979      6558   
sms                       0            0         0       63   4516         0   

scam_type_new   unknown  upi_payment  
source                                
fraud_email        1735            2  
job_posting           0            0  
phishing_email        0            0  
sms                 590            0  

scam_type_new   advance_fee  banking_kyc  job_scam  lottery   none  phishing  \
scam_type                                                                      
banking_kyc             883            0         0        0      0         0   
digital_arrest            8            0    

In [4]:
order = ['digital_arrest', 'upi_payment', 'banking_kyc', 'investment', 'advance_fee', 'lottery']
RULES = sorted(RULES, key=lambda r: order.index(r[0]))
df['scam_type_new'] = df.apply(assign_type, axis=1)

print(pd.crosstab(df['source'], df['scam_type_new']))

for cat in ['upi_payment', 'banking_kyc']:
    print(f"\n===== real rows tagged {cat} =====")
    for m in df[df['scam_type_new'] == cat]['message_decoded']:
        print('-', str(m)[:250].replace('\n', ' '))

print("\n===== fraud_email rows left as unknown =====")
unk = df[(df['source'] == 'fraud_email') & (df['scam_type_new'] == 'unknown')]
for m in unk.sample(12, random_state=3)['message_decoded']:
    print('-', str(m)[:200].replace('\n', ' '))

scam_type_new   advance_fee  banking_kyc  job_scam  lottery   none  phishing  \
source                                                                         
fraud_email            2581            4         0       12   5913         0   
job_posting               0            0       718        0  15348         0   
phishing_email            0            0         0        0  10979      6558   
sms                       0            0         0       63   4516         0   

scam_type_new   unknown  upi_payment  
source                                
fraud_email        1735            2  
job_posting           0            0  
phishing_email        0            0  
sms                 590            0  

===== real rows tagged upi_payment =====
- I am MR. KIMAEVA LIOUDMILA,the personal accountant to Mikhail Khodorkovskythe richest Person in Russia and owner of the following companies: ChairmanCEO: YUKOS OIL (Russian Largest OilCompany) Chairman CEO: Menatep SBP Bank (A well reputable

In [5]:
old = {cat: pats for cat, _, pats in RULES}

REAL_RULES = [
    ('phishing', 2, [r'https?://', r'<a href', r'log ?on\b|log ?in\b|sign ?in\b',
        r'verify your (identity|account|information|details)', r'password',
        r'update your (account|information|details)']),
    ('advance_fee', 2, old['advance_fee']),
    ('lottery', 2, old['lottery']),
]

def assign_type2(row):
    if row['label'] != 'scam':
        return 'none'
    if row['source'] == 'phishing_email':
        return 'phishing'
    if row['source'] == 'job_posting':
        return 'job_scam'
    text = row['message_decoded'].lower()
    for cat, min_hits, pats in REAL_RULES:
        if sum(bool(re.search(p, text)) for p in pats) >= min_hits:
            return cat
    return 'unknown'

df['scam_type_new'] = df.apply(assign_type2, axis=1)
print(pd.crosstab(df['source'], df['scam_type_new']))

real = df[df['source'].isin(['fraud_email', 'sms'])]

sub = real[real['scam_type_new'] == 'phishing']
print(f"\n== phishing rows from fraud_email/sms: {len(sub)} ==")
for m in sub.head(6)['message_decoded']:
    print('-', str(m)[:180].replace('\n', ' '))

unk_f = real[(real['source'] == 'fraud_email') & (real['scam_type_new'] == 'unknown')]
print(f"\n== fraud_email still unknown: {len(unk_f)} ==")
for m in unk_f.sample(10, random_state=5)['message_decoded']:
    print('-', str(m)[:180].replace('\n', ' '))

unk_s = real[(real['source'] == 'sms') & (real['scam_type_new'] == 'unknown')]
print(f"\n== sms still unknown: {len(unk_s)} ==")
for m in unk_s.sample(8, random_state=5)['message_decoded']:
    print('-', str(m)[:180].replace('\n', ' '))

scam_type_new   advance_fee  job_scam  lottery   none  phishing  unknown
source                                                                  
fraud_email            2923         0        8   5913       283     1120
job_posting               0       718        0  15348         0        0
phishing_email            0         0        0  10979      6558        0
sms                       0         0       63   4516         0      590

== phishing rows from fraud_email/sms: 283 ==
- ct of your assistance and account provide for the tranfer, 55% will be my s=hare being the pioneer of the business while the rest 10% will be used to i=nstall a joint company in yo
- <html><div style='background-color:'><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV class=RTE><DI
- <DIV><DIV><DIV>Dear Sir,<BR><BR>I am&nbsp; Ahmed Usman,Bank Manager of Union Bank PLC,Lagos<BR>Branch.I got your contact from the world trade cente

In [6]:
STRONG_419 = [
    r'central bank of nigeria', r'federal republic of nigeria', r'\bnnpc\b',
    r'last wish', r'dying|diagnosed with|cancer', r'\bbeneficiary\b',
    r'solicitor|barrister|attorney|\besq\b',
    r'telephone and fax|phone and fax', r'next of kin', r'\bmy late\b',
    r'joint (business|venture)', r'do business with you',
    r'funds?.{0,40}transfer', r'transfer.{0,40}funds?',
]

PRIZE = [r'\bwon\b', r'\bwinner\b', r'\bwin\b', r'winnings?', r'prize', r'jackpot',
         r'claim (your|ur|yr)\b|claim code', r'awarded', r'congratulations',
         r'free entry', r'\bcomp(etition)?\b']

REAL_RULES = [
    ('advance_fee', 1, STRONG_419),
    ('advance_fee', 2, old['advance_fee']),
    ('lottery', 1, PRIZE),
]

df['scam_type_new'] = df.apply(assign_type2, axis=1)
print(pd.crosstab(df['source'], df['scam_type_new']))

real = df[df['source'].isin(['fraud_email', 'sms'])]

def show(title, frame, n):
    print(f"\n== {title}: {len(frame)} ==")
    for m in frame.sample(min(n, len(frame)), random_state=7)['message_decoded']:
        print('-', str(m)[:170].replace('\n', ' '))

show('sms lottery', real[(real['source'] == 'sms') & (real['scam_type_new'] == 'lottery')], 8)
show('fraud_email lottery', real[(real['source'] == 'fraud_email') & (real['scam_type_new'] == 'lottery')], 8)
show('fraud_email unknown', real[(real['source'] == 'fraud_email') & (real['scam_type_new'] == 'unknown')], 10)
show('sms unknown', real[(real['source'] == 'sms') & (real['scam_type_new'] == 'unknown')], 8)

scam_type_new   advance_fee  job_scam  lottery   none  phishing  unknown
source                                                                  
fraud_email            3409         0       63   5913         0      862
job_posting               0       718        0  15348         0        0
phishing_email            0         0        0  10979      6558        0
sms                       0         0      178   4516         0      475

== sms lottery: 178 ==
- For your chance to WIN a FREE Bluetooth Headset then simply reply back with \ADP\""
- Last Chance! Claim ur a£150 worth of discount vouchers today! Text SHOP to 85023 now! SavaMob, offers mobile! T Cs SavaMob POBOX84, M263UZ. a£3.00 Sub. 16
- ree entry in 2 a weekly comp for a chance to win an ipod. Txt POD to 80182 to get entry (std txt rate) T&C's apply 08452810073 for details 18+
- u r subscribed 2 TEXTCOMP 250 wkly comp. 1st wk?s free question follows, subsequent wks charged@150p/msg.2 unsubscribe txt STOP 2 84128,custcare 087

In [7]:
import base64

B64_CHARS = re.compile(r'[A-Za-z0-9+/=\s]+')

def looks_b64(s):
    s = s.strip()
    return len(s) > 80 and ' ' not in s[:100] and bool(B64_CHARS.fullmatch(s))

def try_b64(text):
    s = str(text).strip()
    if not looks_b64(s):
        return text
    try:
        raw = re.sub(r'\s', '', s)
        raw += '=' * (-len(raw) % 4)
        out = base64.b64decode(raw).decode('utf-8', errors='ignore')
        printable = sum(c.isprintable() or c in '\r\n\t' for c in out) / max(len(out), 1)
        return out if printable > 0.95 and len(out) > 20 else text
    except Exception:
        return text

def decode_email_text_v4(text):
    return decode_email_text_v3(try_b64(text))

b64_rows = df['message'].apply(lambda s: looks_b64(str(s)))
df['message_decoded'] = df['message'].apply(decode_email_text_v4)

fail = df[b64_rows & df['message_decoded'].apply(lambda s: looks_b64(str(s)))]
print("Rows that looked like base64:", b64_rows.sum())
print("Still undecoded:", len(fail))
if len(fail):
    print(repr(str(fail.iloc[0]['message'])[:200]))

STRONG_419 += [r'over-?invoiced', r'ministr(y|ies)', r'abacha', r'royal family',
               r'entitled to receive', r'united nations', r'international monetary fund']
JOB_EMAIL = [r'work (online )?from home', r'get paid weekly', r'job position',
             r'part[- ]time (job|position)']
PRIZE_EMAIL = [p for p in PRIZE if 'awarded' not in p] + [r'sweepstake', r'\blotto\b|lottery']

RULES_EMAIL = [('advance_fee', 1, STRONG_419), ('advance_fee', 2, old['advance_fee']),
               ('job_scam', 1, JOB_EMAIL), ('lottery', 1, PRIZE_EMAIL)]
RULES_SMS = [('lottery', 1, PRIZE)]

def assign_type3(row):
    if row['label'] != 'scam':
        return 'none'
    if row['source'] == 'phishing_email':
        return 'phishing'
    if row['source'] == 'job_posting':
        return 'job_scam'
    text = row['message_decoded'].lower()
    for cat, k, pats in (RULES_SMS if row['source'] == 'sms' else RULES_EMAIL):
        if sum(bool(re.search(p, text)) for p in pats) >= k:
            return cat
    return 'unknown'

df['scam_type_new'] = df.apply(assign_type3, axis=1)
print(pd.crosstab(df['source'], df['scam_type_new']))

emails = df[df['source'] == 'fraud_email']
for name in ['lottery', 'unknown']:
    sub = emails[emails['scam_type_new'] == name]
    print(f"\n== fraud_email {name}: {len(sub)} ==")
    for m in sub.sample(min(10, len(sub)), random_state=11)['message_decoded']:
        print('-', str(m)[:170].replace('\n', ' '))

Rows that looked like base64: 17
Still undecoded: 5
'/9j/4AAQSkZJRgABAQEAlgCWAAD/4QAWRXhpZgAASUkqAAgAAAAAAAAAAAD/2wBDAAEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQH/2wBDAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEB'
scam_type_new   advance_fee  job_scam  lottery   none  phishing  unknown
source                                                                  
fraud_email            3452         6       54   5913         0      822
job_posting               0       718        0  15348         0        0
phishing_email            0         0        0  10979      6558        0
sms                       0         0      178   4516         0      475

== fraud_email lottery: 54 ==
- DearPlease let this letter not to bring confusion to you. I know that we have not met either before but hear me from the dept of your mind. It is circumstances that made 
- be for us to have the firm direct the funds to anotherbank with you as account holder. This way there

In [8]:
LEAD_B64 = re.compile(r'^\s*[A-Za-z0-9+/\s]{80,}={0,2}')

def try_b64_v2(text):
    s = str(text)
    if ' ' in s.lstrip()[:60]:            # real text has spaces early; base64 doesn't
        return text
    m = LEAD_B64.match(s)
    if not m:
        return text
    raw = re.sub(r'[^A-Za-z0-9+/]', '', m.group(0))
    raw = raw[: len(raw) - (len(raw) % 4)]      # drop a trailing partial group
    try:
        out = base64.b64decode(raw).decode('utf-8', errors='ignore')
    except Exception:
        return text
    printable = sum(c.isprintable() or c in '\r\n\t' for c in out) / max(len(out), 1)
    return (out + ' ' + s[m.end():]) if printable > 0.95 and len(out) > 20 else text

def is_image_b64(s):
    s = str(s).lstrip()
    return len(s) > 200 and ' ' not in s[:100] and s.startswith(('/9j/', 'iVBOR', 'R0lGOD'))

def decode_email_text_v5(text):
    return decode_email_text_v3(try_b64_v2(text))

df['is_image'] = df['message'].apply(is_image_b64)
df['message_decoded'] = df['message'].apply(decode_email_text_v5)

still_b64 = df[~df['is_image'] & df['message_decoded'].apply(
    lambda s: ' ' not in str(s).lstrip()[:60] and len(str(s)) > 200)]
print("Image-only emails:", df['is_image'].sum())
print("Text rows still looking like base64:", len(still_b64))
for m in still_b64.head(3)['message']:
    print(repr(str(m)[-150:]))      # the END of the message, to see what follows the body

STRONG_LOTTERY = [r'lott(o|ery)', r'sweepstake', r'takarakuji', r'prize award',
                  r'winnings?', r'lucky winner']
WEAK_PRIZE = [r'\bwon\b', r'\bwinner\b', r'prize', r'congratulations',
              r'claim (your|ur|yr)\b|claim code', r'jackpot']
STRONG_419 += [r'direct the funds', r'account ?holder', r'former president',
               r'security (firm|company)', r'transfer out', r'million (united states )?dollars',
               r'usd ?\d+ ?m\b', r'(may|might) be surpris', r'come to you as a surprise',
               r'since we don.?t know each other', r'honou?rable contractor']

RULES_EMAIL = [('lottery', 1, STRONG_LOTTERY), ('advance_fee', 1, STRONG_419),
               ('advance_fee', 2, old['advance_fee']), ('job_scam', 1, JOB_EMAIL),
               ('lottery', 2, WEAK_PRIZE)]

df['scam_type_new'] = df.apply(assign_type3, axis=1)
print(pd.crosstab(df['source'], df['scam_type_new']))

emails = df[(df['source'] == 'fraud_email') & ~df['is_image']]
for name in ['lottery', 'unknown']:
    sub = emails[emails['scam_type_new'] == name]
    print(f"\n== fraud_email {name}: {len(sub)} ==")
    for m in sub.sample(min(8, len(sub)), random_state=13)['message_decoded']:
        print('-', str(m)[:170].replace('\n', ' '))

Image-only emails: 5
Text rows still looking like base64: 178
'bscribe from this group, send an email to:\nforteana-unsubscribe@egroups.com Your use of Yahoo! Groups is subject to http://docs.yahoo.com/info/terms/ '
'_______________\nSpamassassin-talk mailing list\nSpamassassin-talk@lists.sourceforge.net\nhttps://lists.sourceforge.net/lists/listinfo/spamassassin-talk\n'
'n materials) to their\nexisting library, at a fixed cost most libraries can afford. This should be\ncompelling to small libraries in remote areas.- Jim\n'
scam_type_new   advance_fee  job_scam  lottery   none  phishing  unknown
source                                                                  
fraud_email            3583         6       29   5913         0      716
job_posting               0       718        0  15348         0        0
phishing_email            0         0        0  10979      6558        0
sms                       0         0      178   4516         0      475

== fraud_email lottery: 29 ==
- 

In [9]:
# Confirm the base64 decoder is doing real work
b64_fixed = df['message'].apply(lambda s: try_b64_v2(s) != s).sum()
print("Messages decoded from base64:", b64_fixed)

# Freeze: email prize rows -> unknown (lottery = SMS-only)
mask = (df['source'] == 'fraud_email') & (df['scam_type_new'] == 'lottery')
df.loc[mask, 'scam_type_new'] = 'unknown'

# Drop image-only emails (no text to learn from)
df = df[~df['is_image']].copy()

short = df[(df['source'] == 'fraud_email') & (df['label'] == 'scam') &
           (df['message_decoded'].str.len() < 60)]
print("Very short fraud_email 'scam' fragments (possible label noise):", len(short))

# Save
df['scam_type_old'] = df['scam_type']
df['scam_type'] = df['scam_type_new']
df.drop(columns=['scam_type_new', 'is_image']).to_csv(
    '../data/processed/master_dataset_relabeled.csv', index=False)

print(pd.crosstab(df['source'], df['scam_type']))
print(df['scam_type'].value_counts())

Messages decoded from base64: 84
Very short fraud_email 'scam' fragments (possible label noise): 27
scam_type       advance_fee  job_scam  lottery   none  phishing  unknown
source                                                                  
fraud_email            3583         6        0   5913         0      740
job_posting               0       718        0  15348         0        0
phishing_email            0         0        0  10979      6558        0
sms                       0         0      178   4516         0      475
scam_type
none           36756
phishing        6558
advance_fee     3583
unknown         1215
job_scam         724
lottery          178
Name: count, dtype: int64


#### Synthetic Data ####

In [10]:
synthetic_banking_kyc = [
    "Dear customer, your SBI account will be blocked today. Update your KYC immediately using this link: [SUSPICIOUS_LINK]",

    "HDFC Alert: Your PAN is not linked to your account. Complete verification within 24 hours to avoid account suspension.",

    "Aapka bank account aaj band ho jayega. KYC update karne ke liye abhi [SUSPICIOUS_LINK] par click karein.",

    "Your debit card has been temporarily blocked due to incomplete KYC. Enter your card details here to reactivate it: [SUSPICIOUS_LINK]",

    "RBI KYC NOTICE: Your account verification is pending. Submit your Aadhaar and PAN details immediately to prevent account freezing.",

    "Dear SBI customer, your account verification has failed. Reply with your PAN number and the OTP sent to your registered mobile to restore access.",

    "HDFC BANK: Final reminder! Your KYC expires today. Call our verification department at [SUSPICIOUS_PHONE] to prevent permanent account closure.",

    "ICICI customer alert!!! Your account is on hold. Click [SUSPICIOUS_LINK] and enter your banking details to unlock it now.",

    "Sir, aapki KYC reject ho gayi hai. Account chalu rakhna hai to abhi [SUSPICIOUS_PHONE] par call karke verification complete karwao.",

    "Your Axis Bank account has been selected for mandatory re-verification. Submit your PAN, Aadhaar number and card details through [SUSPICIOUS_LINK].",

    "PNB ALERT: Your account will be deactivated tonight. Send your registered mobile number and verification OTP to complete the KYC process.",

    "Dear customer, your bank account is temporarily restricted. Verify your identity now to restore UPI and debit card services: [SUSPICIOUS_LINK]",

    "Aapka SBI account block hone wala hai. Verification ke liye jo OTP aaya hai woh abhi reply karein.",

    "HDFC KYC Department: Your registered mobile number is not verified. Click [SUSPICIOUS_LINK] and enter your account information to avoid suspension.",

    "URGENT BANK NOTICE!!! PAN verification failed. Send your PAN number immediately to [SUSPICIOUS_PHONE] or your account will be frozen.",

    "ICICI Bank: Your KYC application is incomplete. Pay the required verification fee of ₹499 to activate your account again.",

    "Dear user, your banking profile has expired. Submit your Aadhaar details and debit card PIN through our verification portal today.",

    "Axis Bank Alert: Your account will lose online banking access within 2 hours. Complete the pending KYC form at [SUSPICIOUS_LINK].",

    "PNB customer, aapka account verify nahi hua hai. Apna PAN aur Aadhaar number WhatsApp par bhejkar KYC complete karein.",

    "Your SBI account has been flagged for KYC renewal. Call [SUSPICIOUS_PHONE] now and provide the OTP to prevent account closure.",

    "HDFC ACCOUNT WARNING: Your identity documents have expired. Re-upload your Aadhaar and PAN using [SUSPICIOUS_LINK] within 30 minutes.",

    "Sir/Madam, your ICICI account is pending final verification. Please reply with your customer ID, debit card number and registered mobile number.",

    "Aapka bank account freeze hone wala hai. KYC ke naam par verification pending hai, turant [SUSPICIOUS_LINK] open karke details bharein.",

    "Axis Bank security team: Your account verification failed due to a PAN mismatch. Enter your PAN and banking password at [SUSPICIOUS_LINK] to fix the issue.",

    "PNB FINAL WARNING: Your account will be permanently suspended if you do not submit your identity documents before midnight.",

    "Dear customer, your KYC renewal request has been rejected. Pay ₹299 for immediate re-verification and restore your banking facilities.",

    "SBI ALERT: Your debit card services will stop today. Confirm your card number, expiry date and CVV through [SUSPICIOUS_LINK].",

    "HDFC: Aapka PAN bank account se link nahi hai. Link complete karne ke liye OTP aur account details abhi share karein.",

    "Your ICICI account is pending RBI verification. Download the attached KYC form and provide your personal banking information to continue using your account.",

    "Axis Bank notice: Your account has been temporarily locked. Contact our account verification officer at [SUSPICIOUS_PHONE] and provide the requested OTP.",

    "PNB KYC UPDATE: Your account will be disabled unless you complete the verification form today. Click here: [SUSPICIOUS_LINK]",

    "Dear SBI user, your account has failed the latest KYC check. Enter your Aadhaar number, PAN and ATM PIN to restore transactions.",

    "HDFC customer service: Your bank account requires urgent reactivation. Pay the ₹399 processing charge and submit your account details now.",

    "ICICI BANK ALERT!!! Your account access expires in 15 minutes. Verify your identity using this link immediately: [SUSPICIOUS_LINK]",

    "Aapki KYC pending hai bhai, bank account band ho jayega. Jaldi se OTP aur PAN details bhejo, main update kar deta hoon.",

    "Axis Bank: Your customer profile could not be verified. Reply with your date of birth, PAN and card information to avoid account restrictions.",

    "PNB account holder, your KYC update is overdue. Call [SUSPICIOUS_PHONE] immediately and follow the agent's instructions to keep your account active.",

    "SBI Re-KYC notice: Your account is scheduled for suspension. Complete the online verification and enter the OTP received on your mobile to cancel the suspension.",

    "HDFC BANK FINAL NOTICE: Verification deadline approaching. Submit your account credentials through [SUSPICIOUS_LINK] to prevent permanent deactivation.",

    "Your bank account will be blocked shortly. Send your PAN number and the latest SMS verification code to [SUSPICIOUS_PHONE] for immediate activation."
]

In [13]:
synthetic_digital_arrest = [
    "This is Inspector Rao from Mumbai Cyber Crime. A parcel registered in your name contains illegal items. You must remain on this video call until our investigation is complete.",

    "CBI NOTICE: An arrest warrant has been issued against your Aadhaar number. Do not disconnect this call or inform any family member. Follow my instructions carefully.",

    "Your package has been seized by Customs due to suspected narcotics. Transfer the required verification amount to [SUSPICIOUS_ACCOUNT] to avoid immediate arrest.",

    "This is the TRAI enforcement department. Your mobile number is linked to illegal activities. Press 1 now and cooperate with the investigating officer.",

    "We are investigating your involvement in a money laundering case. Do not leave your residence or switch off your camera until I give you permission.",

    "Main CBI officer bol raha hoon. Aapke Aadhaar par criminal case register hua hai. Call disconnect mat karna, warna arrest warrant issue ho jayega.",

    "Your bank account has been linked to a major cybercrime investigation. I am an officer from the Enforcement Directorate. Keep this matter confidential and answer my questions.",

    "POLICE ALERT: We have found suspicious financial transactions under your identity. Join the video call immediately for verification or face legal action.",

    "Sir, your name has appeared in an international fraud investigation. Send your Aadhaar and PAN details now to avoid being taken into custody.",

    "This is an officer from Delhi Police. A criminal parcel has been found using your address. Stay on the call and do not contact anyone until questioning is finished.",

    "Aapke naam par illegal parcel mila hai. Main Customs Department se bol raha hoon. Abhi video call join karo aur kisi ko phone mat karna.",

    "Your Aadhaar has been used to open multiple suspicious bank accounts. This is a confidential CBI investigation. Failure to cooperate may result in immediate arrest.",

    "We have evidence linking your mobile number to a narcotics shipment. Keep your camera on and do not discuss this investigation with your family.",

    "This is the Enforcement Directorate. Your financial records are under investigation. Transfer ₹25,000 for the mandatory verification process to prevent further legal action.",

    "Attention! A court warrant has been issued in your name. Remain at your current location and follow the instructions given by our investigation team.",

    "Aap digital arrest mein hain. Police investigation chal rahi hai. Ghar se bahar mat nikliye aur kisi ko bhi is call ke baare mein mat batayiye.",

    "This is a senior officer from Mumbai Police. Your identity has surfaced in a serious cybercrime case. You must complete confidential questioning on this video call.",

    "CBI INVESTIGATION: Your bank account is suspected of receiving illegal funds. Provide your account information and remain available for further instructions.",

    "Your passport has been linked to an illegal international shipment. I am from the Customs Department. Cooperate immediately or a warrant will be issued.",

    "This is an official investigation from the cybercrime division. Your name appears in a money laundering report. Do not disconnect, and do not tell your lawyer or relatives about this call.",

    "Sir, main Enforcement Directorate se bol raha hoon. Aapke account mein illegal paisa aaya hai. Investigation complete hone tak call par bane rahiye.",

    "Your SIM card is being investigated for criminal activity. This call is being transferred to a police officer. Follow all instructions to avoid immediate disconnection and legal consequences.",

    "We have detected financial transactions connected to a fraud network. I am an investigating officer. Send your bank statement and identity documents immediately.",

    "Aapke khilaf arrest warrant ready hai. Agar aap cooperate nahi karenge to police team aapke ghar pahunch jayegi. Video call abhi join karo.",

    "This is the Cyber Crime Investigation Bureau. Your identity has been linked to a suspicious bank account. You are required to remain on this call for official questioning.",

    "Your Aadhaar number has appeared in a criminal investigation involving ₹40 lakh. Do not disconnect this call. We need your personal and banking details immediately.",

    "Customs officer speaking. A parcel containing illegal substances was found under your name. Pay the clearance amount of ₹18,000 now to settle the case privately.",

    "This is a confidential police investigation. You must not tell your parents, friends or colleagues about this case. Keep your phone switched on and follow my instructions.",

    "Aapka naam money laundering case mein aaya hai. Main CBI se hoon. Verification ke liye bank details bhejiye aur video call disconnect mat kijiye.",

    "Your mobile number is connected to a serious national security investigation. We are recording this conversation. Remain available until the investigation officer releases you.",

    "ED OFFICER HERE: We have identified suspicious deposits in your account. You must cooperate with our investigation and provide the requested financial records immediately.",

    "You are suspected of participating in an online fraud network. I am a police inspector. Do not leave your home while this investigation is underway.",

    "Aapke Aadhaar ka use karke illegal bank account khola gaya hai. Aapko abhi police verification karwana hoga. Kisi aur ko call mat karna.",

    "This is a CBI officer calling regarding an urgent criminal investigation. Your identity is under suspicion. Stay on the video call and provide your identification documents.",

    "We have intercepted a parcel containing prohibited materials registered in your name. Transfer ₹12,500 to [SUSPICIOUS_ACCOUNT] for case verification and settlement.",

    "A police case has been registered against your mobile number. If you disconnect this call, we will record it as non-cooperation and initiate further legal proceedings.",

    "Main Mumbai Cyber Cell se bol raha hoon. Aapka bank account cyber fraud mein use hua hai. Abhi apni details do aur call kisi ko transfer mat karna.",

    "Your identity is involved in a confidential investigation by the Enforcement Directorate. Keep this matter secret and remain on video until further notice.",

    "This is an urgent customs investigation. Your parcel contains suspected illegal goods. Provide your Aadhaar number and follow the instructions to avoid arrest.",

    "DIGITAL ARREST ORDER: You are required to remain available for questioning. Do not contact anyone, leave your room, or terminate this video call until I authorize it."
    
    "NOTICE NO. CBI/CYB/2026/8812: You are summoned in connection with FIR No. 214/2026 (Money Laundering). You must attend the online hearing at 4 PM today. Failure to appear will result in immediate arrest. This notice is confidential. Do not share it with any family member.",

    "Forwarded as received: ED ALERT. Your Aadhaar number has appeared in a money laundering investigation. Verify your identity immediately. If you do not respond within 2 hours, your accounts will be frozen and an arrest warrant will be issued. Ref: ED/MUM/2041.",

    "CUSTOMS SMS SUMMONS: A parcel registered in your name has been detained at the airport. Your identity is linked to a criminal case. Complete verification today to avoid arrest and legal action. Case Ref: CUS/DEL/5731.",

    "URGENT: PDF WARRANT ATTACHED. An arrest order has been issued against you under Case No. POL/CYB/6284. Open the attached file named Arrest_Order.pdf and complete the required verification immediately. Failure to comply will lead to police action.",

    "TRAI FINAL NOTICE: Your mobile number has been linked to illegal activities and a cybercrime investigation. Your SIM will be disconnected within 2 hours unless you complete identity verification. A police complaint and arrest warrant may follow. Ref: TRAI/26/4817.",

    "NARCOTICS BUREAU CASE NOTICE: Your Aadhaar details are required for verification in connection with Case No. NCB/MUM/3072. You must submit your identity documents and attend the scheduled online investigation today. Do not discuss this matter with anyone until verification is complete.",

    "MUMBAI POLICE CYBER CELL: Your mobile number has been linked to a cybercrime complaint. Submit your Aadhaar and bank details for immediate verification to prevent an FIR from being registered against you. Complaint No: MCC/2026/5291.",

    "WhatsApp Alert — Forwarded as received: Your name has been added to a financial fraud investigation. An arrest warrant is ready for execution. Complete online verification within 60 minutes to avoid detention. Keep this matter confidential and do not inform your relatives. Ref: FIN/DEL/7183.",

    "E-SUMMONS No. ED/BLR/2026/4196: You are required to appear for questioning regarding suspicious financial transactions associated with your identity. Attend the online session at 3:30 PM today. Failure to comply will result in an arrest warrant and freezing of your bank accounts.",

    "Aapka naam ek financial crime investigation mein aaya hai. Aapke khilaf digital arrest notice issue ho gaya hai. Aaj hi identity verification complete karein, warna arrest warrant jari kiya jayega. Ref: HG/8810. Kisi ko is baare mein na batayein."
]

In [14]:
import random, re
random.seed(42)

def fake_url():
    dom = random.choice(['sbi-kycverify.co.in', 'hdfc-secure-update.in', 'icici-kyc-portal.com',
                         'axis-reverify.net', 'pnb-kycupdate.in', 'bank-verify-now.in'])
    return 'http://' + dom + random.choice(['/verify', '/kyc', '/update', '/login', '/secure'])

def fake_phone():
    return '+91' + random.choice('6789') + ''.join(random.choices('0123456789', k=9))

def fake_account():
    return 'A/C ' + ''.join(random.choices('0123456789', k=12))

def fill_placeholders(msg):
    msg = re.sub(r'\[SUSPICIOUS_LINK\]', lambda m: fake_url(), msg)
    msg = re.sub(r'\[SUSPICIOUS_PHONE\]', lambda m: fake_phone(), msg)
    msg = re.sub(r'\[SUSPICIOUS_ACCOUNT\]', lambda m: fake_account(), msg)
    return msg

synthetic_banking_kyc = [fill_placeholders(m) for m in synthetic_banking_kyc]
synthetic_digital_arrest = [fill_placeholders(m) for m in synthetic_digital_arrest]

print("Placeholders left:", sum('[SUSPICIOUS' in m for m in synthetic_banking_kyc + synthetic_digital_arrest))
print(synthetic_banking_kyc[0])
print(synthetic_banking_kyc[6])

Placeholders left: 0
Dear customer, your SBI account will be blocked today. Update your KYC immediately using this link: http://bank-verify-now.in/verify
HDFC BANK: Final reminder! Your KYC expires today. Call our verification department at +917768040250 to prevent permanent account closure.


#### Challenge set ####

In [15]:
import pandas as pd

challenge = {
 'banking_kyc': [
  "Kotak Mahindra Bank: your KYC is incomplete and net banking will be deactivated from tomorrow. Update now at kotak-kycrenew.in/u or call 8847201936",
  "bank of baroda customer, ur account is going to be suspended coz PAN aadhaar link nahi hai. click bit.ly/3kQ9xPz to link, last date today",
  "Canara Bank alert: Video KYC pending for your savings account. Our executive will call you shortly. Kindly share the 6 digit code you receive so the verification can be completed. Account will be restricted otherwise.",
  "Your IndusInd Bank credit card will be blocked at 11:59 PM tonight due to expired KYC. To avoid this, log in at indusind-secure-kyc.com and re-enter your card number, expiry and CVV.",
  "Union Bank of India: dear account holder, as per new RBI guidelines all accounts need re-KYC by 30 Nov. Accounts without it will be marked dormant. Submit documents on www.unionbank-rekyc.in",
  "Yes Bank: Suspicious login detected on your netbanking. Your account is on temporary hold. Verify identity: yesbank-verify.co.in/auth",
  "Hello sir, I am calling from IDFC First Bank KYC department. Your account KYC has expired, kindly tell me your Aadhaar number and the OTP on your phone so I can update it right now.",
  "BoI ALERT: Your a/c ending 4471 will be closed tomorrow. KYC not updated. Pay Rs 10 verification fee and submit PAN details here: bit.ly/4mHt2Rw",
  "PAN card not linked with Aadhaar - your Federal Bank account will be freezed from 1st. Update immediately: federalbank-panlink.in/update",
  "Dear Sir/Madam, Greetings from Punjab & Sind Bank. Your KYC documents have been rejected due to image mismatch. Re-upload a selfie holding your PAN and Aadhaar together at the link below within 24 hours to prevent closure. psb-kycdocs.in/reupload",
  "Sir aapka Bandhan Bank account KYC expire ho gaya hai. Abhi neeche diye link pe jaakar apna account number, PAN aur net banking password dalein varna account block ho jayega. bandhan-kycupdate.in",
  "RBL Bank: Your account KYC will expire in 2 hrs. Failure will lead to freezing of funds. Update at rbl-kycportal.in",
 ],
 'digital_arrest': [
  "Hello, this is Sub-Inspector Verma from Delhi Police Cyber Cell. A FedEx parcel booked with your passport has been caught with 140 grams of MDMA. Do not hang up. Switch on your video and stay where you are, you are now under digital arrest.",
  "FINAL WARNING from Narcotics Control Bureau: Case NCB/DEL/2291 registered against your name. A warrant is ready. To avoid physical arrest connect on Skype ID ncb.officer.delhi now. Do not tell anyone.",
  "Your Aadhaar was used by a trafficking gang to open 11 bank accounts. This is the CBI anti-corruption branch. You are not allowed to disconnect or contact your family until the verification is completed. Cooperate and you will be cleared.",
  "Sir this is Vikram Singh, Customs Mumbai. Your parcel from Taiwan has fake passports and a laptop inside. Case is serious. Keep your phone on and do not talk to anyone, an officer will take your statement on video.",
  "Supreme Court of India - Virtual Hearing Notice: You are directed to appear on video in a money laundering matter. Non-appearance will lead to a non-bailable warrant. Join: scourt-virtual-hearing.in/join Keep this notice confidential.",
  "Aapke SIM se harassment calls ki complaint aayi hai. Main TRAI se bol raha hoon, call ab Mumbai Police ko transfer ho rahi hai. Phone mat kaatna, 2 ghante mein aapka number permanent band aur FIR darj ho jayegi.",
  "This is Inspector Anil Desai. We froze your bank account in a hawala case. To prove your funds are legal you must move your entire balance to the RBI verification account. It will be returned within 24 hours after the check. Remain on the call.",
  "ED notice: A money laundering case (PMLA) has been registered against your PAN. Your assets will be attached. Pay the security deposit of Rs 85,000 to clear your name before 6 PM. Confidential matter, tell nobody including your lawyer.",
  "Cyber crime branch Hyderabad. Your photo and Aadhaar were found at a crime scene. You are placed under house arrest. Do not step outside, keep the video call on 24x7 until the senior officer clears you.",
  "Dear citizen, an Interpol notice has been issued on your passport number. The Passport Office has forwarded it to the CBI. Contact officer R.K. Sharma immediately on 9318274056 to avoid detention at the airport. Do not share this message.",
  "NIA officer here. Your mobile number was used to send threats to a minister. Strictly confidential investigation. Stay on this WhatsApp video call and do not inform your office or your family or you will be treated as an accomplice.",
  "Court order No. 441/CR/2026: You are declared absconding. A police team will reach your home in 30 minutes. To stop this, call the number below and complete verification and settlement. 9265801743",
 ],
 'upi_payment': [
  "Amazon Pay: Your refund of Rs 3,299 for a cancelled order is stuck. Accept the collect request from refund-desk@oksbi and enter your UPI PIN to receive the money.",
  "Wrong transfer: Sir mistakenly sent Rs 20,000 to your PhonePe. Please return it urgently, my mother is in hospital. I have sent a collect request, just approve it.",
  "MobiKwik: Congrats, you won a scratch card of Rs 5,000. Scan the QR code attached and enter your PIN to credit the amount to your bank. Offer valid till 8 PM.",
  "Google Pay: Your UPI ID is being deactivated for non-use. Reply YES and enter your UPI PIN on the link to keep it active: gpay-upi-active.in",
  "CRED: cashback of Rs 1,250 is pending on your bill payment. Click to claim and approve the request using your PIN: bit.ly/3nWq8Yd",
  "Hi, I want to buy the sofa you listed on OLX. I will pay advance. Scan this QR and enter your PIN to receive the payment of Rs 12,000, don't worry money will come to your account directly.",
  "Paytm Payments Bank: Rs 14,999 debited via UPI to unknown merchant. Not you? Call our dispute team immediately on 8120937465 and keep your UPI PIN and OTP ready for reversal.",
  "Electricity bill update: your power will be disconnected tonight 9:30 PM as last month bill not updated. Pay Rs 10 via UPI to the number given and call 7348201695 to verify consumer number.",
  "Sir aapka Paytm wallet KYC expire ho gaya, balance Rs 4,820 freeze ho jayega. Abhi UPI PIN dalkar verify karein: paytm-kycverify.in",
  "Freecharge refund team: Your failed recharge amount Rs 599 will be credited after you approve the pending request on your UPI app. Do not reject the request or the refund will be cancelled.",
  "Dear user, your WhatsApp Pay account shows 2 failed transfers of Rs 6,500. Click the link and enter your debit card details and UPI PIN to reverse the debit. whatsapp-pay-refund.in/reverse",
  "Airtel Thanks: Rs 2,000 cashback on your next recharge. Send Re 1 to the UPI id airtel.cashback@ybl and accept the collect request to activate.",
 ],
 'investment': [
  "Join my Telegram channel for sure-shot stock tips. Last week members made 38% profit in 4 days. SEBI registered analyst. Joining fee only Rs 2,999. t.me/profit_king_official",
  "Congratulations, you have been allotted shares in the upcoming pre-IPO round of a top company at Rs 40 per share. Listing price will be Rs 160. Transfer Rs 50,000 today to secure your allotment.",
  "Dubai gold investment scheme: invest Rs 1 lakh and get Rs 15,000 every month fixed, guaranteed for life. Limited slots. Whatsapp 9810473625 for brochure.",
  "Hello, I'm Priya from the finance team of an international trading firm. My uncle is a senior analyst and gives me inside tips. I started with $200 and now have $18,000. I can show you how, it's very easy 😊",
  "Trade crypto with our AI bot BitProfit. 4% daily returns, withdraw anytime. Deposit USDT now. 2,000 members already earning. Link: bitprofit-ai.com/register",
  "FOREX SIGNALS VIP: 95% accuracy. Invest $100 get $500 in 7 days. Money back guarantee if no profit. Only 20 seats left!!",
  "Sir, we offer real estate plots in Noida with 3x return in 12 months. Booking amount Rs 25,000. Agreement guaranteed buyback. Offer closes Sunday.",
  "Mutual fund guaranteed 24% p.a. returns, no risk, fully tax free. Our scheme has no loss in the last 5 years. Pay Rs 10,000 minimum to start. Call Rajesh 9084216357.",
  "You have been added to the WhatsApp group Stock Millionaires Club. Admin will share a stock tomorrow, buy it at 9:15 and sell at 3 for sure profit. Follow the admin's instructions and fund your trading account at the link.",
  "Start your own business: invest Rs 5,000 in our network marketing plan, recruit just 3 friends and earn Rs 30,000 per month passive income. No work needed. Join before the plan changes on the 1st.",
  "Earn 8% weekly on your savings with our crypto staking platform. 12,000+ happy investors. Minimum Rs 3,000. Withdraw to your bank in minutes. coinstake-india.in",
  "Dear investor, our company is giving fixed deposit at 36% annual interest, much higher than banks. Your principal is 100% safe. Limited to first 50 investors, send payment now to book.",
 ],
}

challenge_df = pd.DataFrame(
    [{'message': m, 'label': 'scam', 'scam_type': t, 'source': 'challenge'}
     for t, msgs in challenge.items() for m in msgs]
)
print(challenge_df['scam_type'].value_counts())
print("duplicates inside set:", challenge_df['message'].duplicated().sum())
challenge_df.to_csv('../data/processed/challenge_set.csv', index=False)

scam_type
banking_kyc       12
digital_arrest    12
upi_payment       12
investment        12
Name: count, dtype: int64
duplicates inside set: 0


#### Legit Training Set ####

In [16]:
import random
import pandas as pd
random.seed(7)

banks = ['SBI', 'HDFC Bank', 'ICICI Bank', 'Axis Bank', 'PNB', 'Kotak Bank', 'Bank of Baroda', 'Canara Bank']
merchants = ['Swiggy', 'Zomato', 'Amazon', 'Flipkart', 'BigBasket', 'IRCTC', 'Uber', 'Myntra', 'Reliance Smart', 'Apollo Pharmacy']
apps = ['PhonePe', 'Google Pay', 'Paytm', 'Amazon Pay', 'Freecharge', 'MobiKwik']
funds = ['Nifty 50 Index Fund', 'Flexi Cap Fund', 'Bluechip Fund', 'Mid Cap Fund']

bank = lambda: random.choice(banks)
merch = lambda: random.choice(merchants)
app = lambda: random.choice(apps)
amt = lambda lo=100, hi=15000: f"{random.randint(lo, hi):,}"
last4 = lambda: str(random.randint(1000, 9999))
date = lambda: f"{random.randint(1, 28):02d}-{random.choice(['Oct', 'Nov', 'Dec', 'Jan', 'Feb'])}"
care = lambda: f"1800 {random.randint(100, 999)} {random.randint(1000, 9999)}"
otp = lambda: str(random.randint(100000, 999999))

templates = [
    lambda: f"{bank()}: Rs {amt()} debited from A/c ending {last4()} on {date()} via UPI to {merch()}. Not you? Call {care()}. Never share your PIN or OTP with anyone.",
    lambda: f"Rs {amt()} credited to your {bank()} A/c ending {last4()} on {date()}. Available balance Rs {amt(1000, 90000)}.",
    lambda: f"{otp()} is your OTP for {random.choice(['login', 'a payment of Rs ' + amt(), 'adding a new payee'])}. Valid for 10 minutes. Do not share this OTP with anyone. {bank()} never asks for it.",
    lambda: f"Your {merch()} order #{random.randint(10000000, 99999999)} is out for delivery today. Delivery OTP is {otp()}. Please share it only with the delivery partner.",
    lambda: f"Refund of Rs {amt()} for your cancelled {merch()} order has been credited to your {bank()} account ending {last4()}. It may take up to 3 working days to reflect.",
    lambda: f"Dear customer, your KYC is due for renewal. Please visit your {bank()} branch with a valid ID and address proof, or update it in the official app. Ignore this message if already updated.",
    lambda: f"Your {bank()} credit card statement is Rs {amt(500, 40000)}. Minimum due Rs {amt(100, 2000)}, due date {date()}. Pay via the app or net banking.",
    lambda: f"Cashback of Rs {amt(10, 300)} has been credited to your wallet for your recent payment. Thanks for using {app()}.",
    lambda: f"Your SIP of Rs {amt(500, 10000)} in {random.choice(funds)} has been processed. Units allotted at NAV Rs {random.randint(20, 400)}.{random.randint(10, 99)}. Thank you for investing.",
    lambda: f"Your electricity bill of Rs {amt(300, 5000)} for consumer no. {random.randint(1000000, 9999999)} is due on {date()}. Pay on the official website or app to avoid late fees.",
    lambda: f"{bank()} advisory: Never share your OTP, PIN or CVV with anyone, including bank staff. We will never ask for these on a call, SMS or WhatsApp. Report cyber fraud on 1930.",
    lambda: f"Recharge of Rs {random.choice([199, 239, 299, 479, 599, 719])} successful for your number. Validity {random.choice([28, 56, 84])} days. Thanks for choosing {app()}.",
]

generated = set()
while len(generated) < 120:
    generated.add(random.choice(templates)())
generated = list(generated)

handwritten_legit = [
    "Bhai kal 5 baje milte hain, paise UPI kar dunga, tab tak thoda wait kar",
    "Mom, I have transferred Rs 5000 to your account, check once. Bank mein dikh jayega 10 min mein",
    "Hi, this is Rahul from Axis Bank. As we discussed, please visit the branch tomorrow with your PAN and Aadhaar for the KYC update. Ask for me at the counter.",
    "Your application for a new SBI debit card has been received. It will be delivered to your registered address in 7 working days. Track the status in the YONO app.",
    "Advocate Mehta: Hearing in case 142/2026 is adjourned to 12 Dec. Please bring the original documents. No need to come before 11 AM.",
    "Mumbai Police advisory: There is no such thing as digital arrest. Police, CBI or customs will never arrest you over a video call or ask you to transfer money. If you receive such a call, hang up and dial 1930.",
    "Income Tax Department: Your refund of Rs 12,450 for AY 2025-26 has been processed and will be credited to your bank account within 10 days. Check status on the official e-filing portal.",
    "SEBI Investor Alert: Do not trust stock tips from unknown Telegram or WhatsApp groups. Invest only through SEBI registered intermediaries and check registration on the SEBI website.",
    "Dear investor, shares have been allotted to you in the recent IPO. Please check your demat account for the credit. For details visit the registrar's website.",
    "yaar maine 5000 ka SIP start kiya hai Nifty index fund mein, tu bhi try kar, long term ke liye accha hai",
    "Hey, are we still on for dinner at 8? Rahul is bringing the cake.",
    "Your Delhivery shipment will be delivered today between 2 PM and 6 PM. Delivery partner Suresh: 9876543210. Please keep the OTP ready.",
    "Jio: Your data pack expires tomorrow. Recharge on the MyJio app to continue enjoying unlimited data.",
    "Salary of Rs 45,200 credited to your HDFC Bank a/c ending 8812 on 30-Nov. Avl bal Rs 61,730.",
    "LIC: Your premium of Rs 8,400 is due on 20 Dec. Pay online at licindia.in or at your nearest branch.",
    "Rahul Sharma has requested Rs 450 from you on PhonePe for dinner split. Open the app to review and pay or decline.",
    "Zomato: Your refund of Rs 249 has been processed to your original payment method. It takes 5-7 working days.",
    "Dear student, the last date for exam form submission is 20 Dec. Pay the fees on the university portal. Late fee applies after this date.",
    "Aapka OTP 482913 hai. Ise kisi ke saath share na karein. Bank kabhi OTP nahi maangta.",
    "Your Aadhaar address update request has been accepted. You can download the updated e-Aadhaar from the official UIDAI website.",
    "Amazon: Your package was delivered to the front door. Tell us how it went by rating the delivery in the app.",
    "Dr. Sen's clinic: Your appointment is confirmed for Tuesday 4 PM. Please carry your previous reports.",
    "Dear customer, as per RBI guidelines your account KYC is due. You can update it by visiting your branch with ID proof, or ignore this message if you have already done so.",
    "Bhai tera Amazon wala refund aaya kya? Mera toh 3 din mein aa gaya tha, bank se check kar.",
]

legit_df = pd.DataFrame(
    [{'message': m, 'label': 'legit', 'scam_type': 'none', 'source': 'synthetic_legit'}
     for m in generated + handwritten_legit]
).drop_duplicates(subset='message')

print(legit_df.shape)
print(legit_df.sample(6, random_state=1)['message'].to_string())
legit_df.to_csv('../data/processed/synthetic_legit.csv', index=False)

(144, 4)
94    Recharge of Rs 599 successful for your number....
91    Your electricity bill of Rs 1,358 for consumer...
84    Your electricity bill of Rs 4,185 for consumer...
48    Rs 2,745 credited to your PNB A/c ending 1825 ...
53    Axis Bank: Rs 8,754 debited from A/c ending 69...
5     Your SIP of Rs 4,017 in Nifty 50 Index Fund ha...


#### Legit Challege set ####

In [17]:
challenge_legit = [
    "HDFC Bank: Rs 1,240.00 spent on Card XX4417 at BigBasket on 03-Nov. Avl limit Rs 84,220. If not you, call 1800 123 4567.",
    "Dear Customer, your Kotak account KYC is up to date. No action needed. Thanks for banking with us.",
    "bhai paytm pe 200 bhej diya check kar le, chai ke paise",
    "Your OTP for login to Groww is 773920. It is valid for 5 min. Don't share with anyone.",
    "Union Bank of India: Your fixed deposit of Rs 50,000 matures on 14-Dec-2026. Please visit your branch or use internet banking to renew or withdraw.",
    "Reminder: Your Star Health policy renews on 02 Jan. Renew online using the Star Health app to avoid a lapse in cover.",
    "RBI has clarified that it never contacts citizens over phone or WhatsApp for KYC or account blocking. Do not click on unknown links. Report cyber fraud at cybercrime.gov.in or call 1930.",
    "Your Ola ride receipt: Rs 187 paid via UPI. Thank you for riding. Rate your driver in the app.",
    "ICICI Bank: Your credit card ending 3302 has been temporarily blocked at your request. To unblock, use the iMobile Pay app or call the number on the back of your card.",
    "Congratulations Rohan on your promotion! Party this Saturday at my place, 7 PM. Bring your own drinks 😄",
    "Dear investor, your KYC status for mutual fund investments is validated. You can now invest using your registered email and mobile. - CAMS",
    "Hi sir, this is Priya from Canara Bank Koramangala branch. Your locker rent of Rs 1,500 is due. Please pay at the branch or via net banking by 30th.",
]

legit_ch = pd.DataFrame(
    [{'message': m, 'label': 'legit', 'scam_type': 'none', 'source': 'challenge'} for m in challenge_legit]
)
challenge_all = pd.concat([pd.read_csv('../data/processed/challenge_set.csv'), legit_ch], ignore_index=True)
challenge_all = challenge_all.drop_duplicates(subset='message')
challenge_all.to_csv('../data/processed/challenge_set.csv', index=False)
print(challenge_all.groupby(['label', 'scam_type']).size())

label  scam_type     
legit  none              12
scam   banking_kyc       12
       digital_arrest    12
       investment        12
       upi_payment       12
dtype: int64


In [19]:
import json, os
os.makedirs('../data/processed/synth_lists', exist_ok=True)

names = ['synthetic_upi', 'synthetic_investment', 'synthetic_banking_kyc',
         'synthetic_digital_arrest', 'synthetic_digital_arrest_written']

for name in names:
    if name in globals():
        data = list(globals()[name])
        with open(f'../data/processed/synth_lists/{name}.json', 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False)
        left = sum('[SUSPICIOUS' in m for m in data)
        print(f"saved {name}: {len(data)} messages, placeholders left: {left}")
    else:
        print(f"{name}: not found in this notebook")

synthetic_upi: not found in this notebook
synthetic_investment: not found in this notebook
saved synthetic_banking_kyc: 40 messages, placeholders left: 0
saved synthetic_digital_arrest: 49 messages, placeholders left: 0
synthetic_digital_arrest_written: not found in this notebook


In [20]:
import json, os
import pandas as pd

def load(name, required=True):
    path = f'../data/processed/synth_lists/{name}.json'
    if not os.path.exists(path):
        if required:
            raise FileNotFoundError(path)
        return []
    with open(path, encoding='utf-8') as f:
        return json.load(f)

digital = list(dict.fromkeys(
    load('synthetic_digital_arrest') + load('synthetic_digital_arrest_written', required=False)
))

scam_lists = {
    'upi_payment': load('synthetic_upi'),
    'investment': list(dict.fromkeys(load('synthetic_investment'))),
    'banking_kyc': load('synthetic_banking_kyc'),
    'digital_arrest': digital,
}

synthetic_scam_df = pd.DataFrame(
    [{'message': m, 'label': 'scam', 'scam_type': t, 'source': 'synthetic'}
     for t, ms in scam_lists.items() for m in ms]
)

print("placeholders left:", synthetic_scam_df['message'].str.contains(r'\[SUSPICIOUS', regex=True).sum())
print(synthetic_scam_df['scam_type'].value_counts())
synthetic_scam_df.to_csv('../data/processed/synthetic_scam.csv', index=False)

placeholders left: 0
scam_type
digital_arrest    49
upi_payment       40
investment        40
banking_kyc       40
Name: count, dtype: int64
